In [1]:
# pandas for data manipulation
# re for regular expressions
import re

# graph helpers
import numpy as np
import pandas as pd

from utils.graphs import plot_magnitude_sets_analysis_interactive
from utils.import_data import get_csv_files, print_user_location_tables, sort_meta_info

In [3]:
path_to: str = "C:\\Users\\pedro\\OneDrive - Universidade de Coimbra\\Ambiente de Trabalho\\tese\\thesis-project"
path: str = path_to + "\\CSI DATA"

# scenario -> location -> user -> esp -> trial -> file_path
FileMap = dict[str, dict[str, dict[str, dict[str, dict[str, str]]]]]
# scenario -> location -> user -> esp -> trial -> csi ndarray
csi_map = dict[str, dict[str, dict[str, dict[str, dict[str, np.ndarray]]]]]

all_data_files = get_csv_files(path)
scenarios_id, locations_id, users_id, esps_id, trials_id = sort_meta_info(path)

print(f"Scenarios present: {', '.join(scenarios_id) or 'none'}")
print_user_location_tables(all_data_files)


Scenarios present: 1

User 01 - Pedro Monteiro
    1  2  3  4  5  6  7  8  9 10 11 12 13 14
A | .  .  .  .  .  .  .  .  .  .  .  .  .  .
B | .  .  .  .  .  .  .  .  .  .  .  .  .  .
C | .  .  .  .  .  .  .  .  .  .  .  .  .  .
D | .  .  .  .  .  .  .  .  .  .  .  .  .  .
E | .  .  .  .  .  .  .  .  .  .  .  .  .  .
F | .  .  .  .  .  .  .  .  .  .  .  .  .  .
No files found for this user.

User 02 - Guilherme Cabaço
    1  2  3  4  5  6  7  8  9 10 11 12 13 14
A | X  X  .  .  X  .  .  .  .  .  .  .  X  X
B | X  X  .  .  X  .  .  .  .  X  X  X  X  X
C | X  X  X  X  X  X  X  X  X  X  X  .  .  X
D | X  X  X  X  X  X  X  X  .  .  .  .  .  .
E | X  X  X  X  X  X  X  X  .  X  X  X  X  .
F | .  .  .  .  X  .  .  X  .  X  X  X  X  .
Other locations: Z-0

User 03 - Henrique
    1  2  3  4  5  6  7  8  9 10 11 12 13 14
A | X  X  .  .  X  .  .  .  .  .  .  .  X  X
B | X  X  .  .  X  .  .  X  .  X  X  X  X  X
C | X  X  X  X  X  X  X  X  X  X  X  .  .  X
D | X  X  X  X  X  X  X  X  .  .  .  .  .  .

In [ ]:
FIVE_GHZ_RAW_LENGTH_114 = 114
FIVE_GHZ_RAW_LENGTH_228 = 228
FIVE_GHZ_ZERO_SUBCARRIER_INDEXES = {
    FIVE_GHZ_RAW_LENGTH_114: [28],
    FIVE_GHZ_RAW_LENGTH_228: [28, 85],
}
FIVE_GHZ_228_SPIKE_THRESHOLD = 50.0


def empty_magnitude(raw_length: int) -> np.ndarray:
    subcarrier_count = raw_length // 2
    removed_subcarriers = len(FIVE_GHZ_ZERO_SUBCARRIER_INDEXES[raw_length])
    return np.empty((0, subcarrier_count - removed_subcarriers), dtype=float)


def csi_values_to_magnitude(csi_values: np.ndarray, raw_length: int) -> np.ndarray:
    if csi_values.size == 0:
        return empty_magnitude(raw_length)

    imag = csi_values[:, ::2]
    real = csi_values[:, 1::2]
    complex_csi = real + 1j * imag
    mag = np.abs(complex_csi)

    # Remove the zero-magnitude/DC subcarrier from each 114-value block.
    return np.delete(mag, FIVE_GHZ_ZERO_SUBCARRIER_INDEXES[raw_length], axis=1)


def remove_228_spike_subcarriers(
    mag_228: np.ndarray,
    mag_228_first_half: np.ndarray,
    mag_228_second_half: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    if mag_228.shape[0] == 0:
        return mag_228, mag_228_first_half, mag_228_second_half

    spike_subcarriers = np.flatnonzero(
        mag_228.mean(axis=0) > FIVE_GHZ_228_SPIKE_THRESHOLD,
    )
    if spike_subcarriers.size == 0:
        return mag_228, mag_228_first_half, mag_228_second_half

    first_half_width = mag_228_first_half.shape[1]
    first_half_spikes = spike_subcarriers[spike_subcarriers < first_half_width]
    second_half_spikes = spike_subcarriers[spike_subcarriers >= first_half_width] - first_half_width

    mag_228 = np.delete(mag_228, spike_subcarriers, axis=1)
    mag_228_first_half = np.delete(mag_228_first_half, first_half_spikes, axis=1)
    mag_228_second_half = np.delete(mag_228_second_half, second_half_spikes, axis=1)

    return mag_228, mag_228_first_half, mag_228_second_half


def remove_spike_subcarriers(magnitude: np.ndarray) -> np.ndarray:
    if magnitude.shape[0] == 0:
        return magnitude

    spike_subcarriers = np.flatnonzero(
        magnitude.mean(axis=0) > FIVE_GHZ_228_SPIKE_THRESHOLD,
    )
    if spike_subcarriers.size == 0:
        return magnitude

    return np.delete(magnitude, spike_subcarriers, axis=1)


def remove_shared_spike_subcarriers(
    first_magnitude: np.ndarray,
    second_magnitude: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    if first_magnitude.shape[0] == 0 and second_magnitude.shape[0] == 0:
        return first_magnitude, second_magnitude

    first_spikes = np.flatnonzero(
        first_magnitude.mean(axis=0) > FIVE_GHZ_228_SPIKE_THRESHOLD,
    ) if first_magnitude.shape[0] else np.array([], dtype=int)
    second_spikes = np.flatnonzero(
        second_magnitude.mean(axis=0) > FIVE_GHZ_228_SPIKE_THRESHOLD,
    ) if second_magnitude.shape[0] else np.array([], dtype=int)
    spike_subcarriers = np.union1d(first_spikes, second_spikes).astype(int)

    if spike_subcarriers.size == 0:
        return first_magnitude, second_magnitude

    return (
        np.delete(first_magnitude, spike_subcarriers, axis=1),
        np.delete(second_magnitude, spike_subcarriers, axis=1),
    )


def split_228_interleaved_parts(csi_228: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    if csi_228.size == 0:
        empty_csi = np.empty((0, FIVE_GHZ_RAW_LENGTH_114), dtype=float)
        return empty_csi, empty_csi.copy()

    grouped_csi = csi_228.reshape(csi_228.shape[0], -1, 4)
    first_part = grouped_csi[:, :, :2].reshape(csi_228.shape[0], -1)
    second_part = grouped_csi[:, :, 2:].reshape(csi_228.shape[0], -1)

    return first_part, second_part


def make_magnitude_map() -> csi_map:
    return {}


def set_magnitude_entry(
    magnitudes: csi_map,
    scenario_key: str,
    location_key: str,
    user_key: str,
    esp_key: str,
    trial_key: str,
    magnitude: np.ndarray,
) -> None:
    magnitudes.setdefault(scenario_key, {})
    magnitudes[scenario_key].setdefault(location_key, {})
    magnitudes[scenario_key][location_key].setdefault(user_key, {})
    magnitudes[scenario_key][location_key][user_key].setdefault(esp_key, {})
    magnitudes[scenario_key][location_key][user_key][esp_key][trial_key] = magnitude


def process_csi(data_file: str) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, int, int, int]:
    # 5 GHz saved CSV header from app_main.c UDP payload: type,seq,mac,rssi,rate,noise_floor,fft_gain,agc_gain,channel,local_timestamp,sig_len,rx_state,len,first_word,data
    file_csv = pd.read_csv(data_file, header=None, usecols=[14])
    csi_raw: pd.Series = file_csv.iloc[:, 0]

    valid_csi_114: list[list[float]] = []
    valid_csi_228: list[list[float]] = []
    no_match_count: int = 0
    no_complete_count: int = 0

    for entry in csi_raw:
        match = re.search(r"\[(.*?)\]", str(entry))
        if not match:
            no_match_count += 1
            continue

        nums = [float(n) for n in re.findall(r"-?\d+", match.group(1))]

        if len(nums) == FIVE_GHZ_RAW_LENGTH_114:
            valid_csi_114.append(nums)
        elif len(nums) == FIVE_GHZ_RAW_LENGTH_228:
            valid_csi_228.append(nums)
        else:
            no_complete_count += 1

    csi_114 = np.array(valid_csi_114, dtype=float)
    csi_228 = np.array(valid_csi_228, dtype=float)

    mag_114 = csi_values_to_magnitude(csi_114, FIVE_GHZ_RAW_LENGTH_114)
    mag_228 = csi_values_to_magnitude(csi_228, FIVE_GHZ_RAW_LENGTH_228)

    if csi_228.size == 0:
        mag_228_first_half = empty_magnitude(FIVE_GHZ_RAW_LENGTH_114)
        mag_228_second_half = empty_magnitude(FIVE_GHZ_RAW_LENGTH_114)
        mag_228_interleaved_first = empty_magnitude(FIVE_GHZ_RAW_LENGTH_114)
        mag_228_interleaved_second = empty_magnitude(FIVE_GHZ_RAW_LENGTH_114)
    else:
        mag_228_first_half = csi_values_to_magnitude(
            csi_228[:, :FIVE_GHZ_RAW_LENGTH_114],
            FIVE_GHZ_RAW_LENGTH_114,
        )
        mag_228_second_half = csi_values_to_magnitude(
            csi_228[:, FIVE_GHZ_RAW_LENGTH_114:],
            FIVE_GHZ_RAW_LENGTH_114,
        )
        mag_228, mag_228_first_half, mag_228_second_half = remove_228_spike_subcarriers(
            mag_228,
            mag_228_first_half,
            mag_228_second_half,
        )

        csi_228_interleaved_first, csi_228_interleaved_second = split_228_interleaved_parts(csi_228)
        mag_228_interleaved_first = csi_values_to_magnitude(
            csi_228_interleaved_first,
            FIVE_GHZ_RAW_LENGTH_114,
        )
        mag_228_interleaved_second = csi_values_to_magnitude(
            csi_228_interleaved_second,
            FIVE_GHZ_RAW_LENGTH_114,
        )
        mag_228_interleaved_first, mag_228_interleaved_second = remove_shared_spike_subcarriers(
            mag_228_interleaved_first,
            mag_228_interleaved_second,
        )

    return (
        mag_114,
        mag_228,
        mag_228_first_half,
        mag_228_second_half,
        mag_228_interleaved_first,
        mag_228_interleaved_second,
        no_match_count,
        no_complete_count,
        len(csi_raw),
    )


def process_files(data_files: FileMap) -> tuple[csi_map, csi_map, csi_map, csi_map, csi_map, csi_map]:
    magnitudes_114 = make_magnitude_map()
    magnitudes_228 = make_magnitude_map()
    magnitudes_228_first_half = make_magnitude_map()
    magnitudes_228_second_half = make_magnitude_map()
    magnitudes_228_interleaved_first = make_magnitude_map()
    magnitudes_228_interleaved_second = make_magnitude_map()
    no_match_count: int = 0
    no_complete_count: int = 0
    total_entries: int = 0

    for scenario_key, locations_map in data_files.items():
        for location_key, users_map in locations_map.items():
            for user_key, esps_map in users_map.items():
                for esp_key, trials_map in esps_map.items():
                    esp_id = int(esp_key.removeprefix("esp_"))
                    if not 11 <= esp_id <= 20:
                        continue

                    for trial_key, file_path in trials_map.items():
                        if file_path is None:
                            continue

                        (
                            mag_114,
                            mag_228,
                            mag_228_first_half,
                            mag_228_second_half,
                            mag_228_interleaved_first,
                            mag_228_interleaved_second,
                            no_match,
                            no_complete,
                            total,
                        ) = process_csi(str(file_path))

                        set_magnitude_entry(
                            magnitudes_114,
                            scenario_key,
                            location_key,
                            user_key,
                            esp_key,
                            trial_key,
                            mag_114,
                        )
                        set_magnitude_entry(
                            magnitudes_228,
                            scenario_key,
                            location_key,
                            user_key,
                            esp_key,
                            trial_key,
                            mag_228,
                        )
                        set_magnitude_entry(
                            magnitudes_228_first_half,
                            scenario_key,
                            location_key,
                            user_key,
                            esp_key,
                            trial_key,
                            mag_228_first_half,
                        )
                        set_magnitude_entry(
                            magnitudes_228_second_half,
                            scenario_key,
                            location_key,
                            user_key,
                            esp_key,
                            trial_key,
                            mag_228_second_half,
                        )
                        set_magnitude_entry(
                            magnitudes_228_interleaved_first,
                            scenario_key,
                            location_key,
                            user_key,
                            esp_key,
                            trial_key,
                            mag_228_interleaved_first,
                        )
                        set_magnitude_entry(
                            magnitudes_228_interleaved_second,
                            scenario_key,
                            location_key,
                            user_key,
                            esp_key,
                            trial_key,
                            mag_228_interleaved_second,
                        )

                        no_match_count += no_match
                        no_complete_count += no_complete
                        total_entries += total

        # print(f"Total invalid CSI entries (no match): {no_match_count}")
        # print(f"Total invalid CSI entries (incomplete): {no_complete_count}")
        # if total_entries > 0:
        #     print(f"Total percentage of no match: {no_match_count / total_entries:.2%}")
        #     print(f"Total percentage of incomplete: {no_complete_count / total_entries:.2%}\n\n")
        # else:
        #     print("No entries processed.\n\n")

        no_match_count = 0
        no_complete_count = 0
        total_entries = 0

    return (
        magnitudes_114,
        magnitudes_228,
        magnitudes_228_first_half,
        magnitudes_228_second_half,
        magnitudes_228_interleaved_first,
        magnitudes_228_interleaved_second,
    )

# Use the importable implementation so multi-core workers work from notebooks on Windows.
from utils.csi_processing import process_228_files as process_files

In [ ]:
(
    magnitude_114_data,
    magnitude_228_data,
    magnitude_228_first_half_data,
    magnitude_228_second_half_data,
    magnitude_228_interleaved_first_data,
    magnitude_228_interleaved_second_data,
) = process_files(all_data_files)


In [ ]:
# Asks what location/ESPs to draw, then shows all 4 magnitude sets in each figure.
magnitude_sets = {
    "114": magnitude_114_data,
    "228": magnitude_228_data,
    "228 first half": magnitude_228_first_half_data,
    "228 second half": magnitude_228_second_half_data,
    "228 interleaved first": magnitude_228_interleaved_first_data,
    "228 interleaved second": magnitude_228_interleaved_second_data,
}

plot_magnitude_sets_analysis_interactive(magnitude_sets)
